# News To Stock Analyser 데이터준비

뉴스기사를 전달하면, 긍/부정분석 뿐아니라, 특정주식에 대한 긍/부정평가 처리 RAG 구현

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")
os.environ['TAVILY_API_KEY'] = os.getenv("tavily_key")
os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN") 
KAKAO_API_KEY = os.getenv("kakao_key")

## 데이터준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [3]:
from datasets import load_dataset

dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")

c:\Users\Playdata\llm\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [5]:
economy_dataset = dataset["train"].filter(lambda row : row["category"] == "economy")
print(len(economy_dataset))

Filter: 100%|██████████| 22194/22194 [00:00<00:00, 67260.39 examples/s]

17088


In [6]:
import pandas as pd

df = economy_dataset.to_pandas()
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## sLLM 답변데이터 생성
llm을 이용해서 sLLM이 답변했으면 하는 내용을 생성해낸다. 이때 답변을 품질이 중요하므로, 되도록 상위모델을 사용하는 것이 좋다.

In [7]:
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate

class StockAnalysis(BaseModel):
    stock_related: bool = Field(description = "뉴스와 주식 종목간의 연관성 여부")
    summary: str = Field(description = "뉴스 요약")

    positive_stocks: List[str] = Field(description = "긍정적 영향이 예상되는 주식 종목명 목록", default_factory = list)
    positive_keywords: List[str] = Field(description = "긍정적 영향의 근거가 되는 키워드 목록", default_factory = list)
    positive_reasons: List[str] = Field(description = "긍정적 영향이 예상되는 이유", default_factory = list)
    
    negative_stocks: List[str] = Field(description = "부정적 영향이 예상되는 주식 종목명 목록", default_factory = list)
    negative_keywords: List[str] = Field(description = "부정적 영향의 근거가 되는 키워드 목록", default_factory = list)
    negative_reasons: List[str] = Field(description = "부정적 영향이 예상되는 이유", default_factory = list)
    


In [8]:
system_prompt = '''  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''

user_prompt = '''  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트
다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', user_prompt)
])

news = df['document'][1]
prompt.invoke({'news': news})

ChatPromptValue(messages=[SystemMessage(content="  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트\n다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.\n\n[news]\n문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션

In [9]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4.1-mini")
chain = prompt | llm.with_structured_output(StockAnalysis) 

def analyze_news(news):
    return chain.invoke({"news" : news})

analyze_news(news)

StockAnalysis(stock_related=False, summary="인터컨티넨탈 서울 코엑스의 뷔페 레스토랑 브래서리가 6월 6일부터 8월 31일까지 '쿨 섬머 페스타'를 진행한다. 대표 메뉴인 미국식 해산물 찜인 시푸드 보일을 비롯해 다양한 해산물 요리를 제공하며, 소믈리에 추천 와인 5종과 생맥주 무제한 제공 옵션도 선택 가능하다. 프로모션 기간 와인 구매자에게는 주트백 증정, 네이버 예약 시 할인 혜택도 제공된다.", positive_stocks=[], positive_keywords=[], positive_reasons=[], negative_stocks=[], negative_keywords=[], negative_reasons=[])

In [10]:
news = df["document"][100]
print(news)
print()

analyze_news(news)

해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관계자의 의견을 사전에 수렴할 수 없는 문제가 있었다. 이에 공유수면 점용·사용으로 인한 사회적 갈등이 증가했다. 이러한 문제를 해결하기 위해 해수부는 지난 1월 공유수면 점용·사용 허가를 할 때 이해관계자의 의견을 듣도록 공유수면 관리 및 매립에 관한 법률을 개정했다. 법 개정에 따라 공유수면관리청이 해양환경·수산자원·자연경관 보호 등에 영향을 끼칠 수 있는 공유수면 점용·사용 신청을 받은 경우 이를 관보 공보 와 인터넷 홈페이지에 공고해야 한다. 또 점용·사용 허가를 했을 때 피해를 볼 것으로 예상되는 어업인에 대한 의견 조사도 별도로 진행해야 한다. 황준성 해수부 해양공간정책과장은 공유수면 점용·사용으로 인한 이해 관계자의 피해를 방지하려는 법령 개정의 취지를 달성할 수 있도록 각 공유수면관리청과 협력해 관련 제도의 차질 없는 운영을 지원하겠다 고 말했다.



StockAnalysis(stock_related=False, summary='해양수산부가 공유수면 관리 및 매립에 관한 법률을 개정하여, 공유수면 점용 및 사용 허가 시 어업인 등 이해관계자의 의견을 사전에 수렴하도록 하는 내용을 포함해 5일부터 시행한다. 이번 법 개정은 공유수면 점용·사용으로 인한 사회적 갈등을 완화하고 해양환경 및 수산자원 보호를 강화하기 위한 조치다. 해수부는 관련 제도의 원활한 운영을 위해 공유수면관리청과 협력할 계획이다.', positive_stocks=[], positive_keywords=[], positive_reasons=[], negative_stocks=[], negative_keywords=[], negative_reasons=[])

In [11]:
df = df[:50]
df["content"] = df["title"] + "\n" + df["document"]

pd.set_option("display.max_colwidth", None)
df["content"].head()

0                                                                                                                                                                      추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 

In [13]:
# LLM 질의 & 데이터 생성
from tqdm.auto import tqdm  # 프로그레스바

results = []    # 분석 결과 저장 리스트

for content in tqdm(df['content']):
    result = analyze_news(content)  # 분석 진행([StockAnalysis]형태로 반환)
    results.append(result)          # 리스트에 추가
    
df['result'] = results              # 결과 리스트를 DF result 컬럼으로 추가
df.head()

100%|██████████| 50/50 [02:14<00:00,  2.68s/it]


,date,category,press,title,document,link,summary,content,result
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,https://n.news.naver.com/mnews/article/052/0001759333?sid=101,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했다.",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,"stock_related=True summary='정부가 수출 확대를 위해 무역금융 규모를 40조 원 이상 확대하고, 물류비 지원과 임시선박 투입 등 수출 중소기업 지원 대책을 추진키로 했습니다. 또한 반도체 등 첨단 산업 육성 전략과 에너지 효율화 방안도 마련해 무역수지 개선에 나설 계획입니다.' positive_stocks=['삼성전자', 'SK하이닉스', '현대글로비스', '팬오션'] positive_keywords=['무역금융 확대', '물류비 지원', '임시선박 투입', '첨단산업 육성', '수출 확대'] positive_reasons=['무역금융 확대와 물류비 절감 대책은 수출 중소기업의 경쟁력 강화에 긍정적 영향', '임시선박 투입으로 물류난 해소 기대', '반도체 등 첨단 산업 육성 전략은 수출 주도 산업 강화와 관련주에 긍정적', '현대글로비스, 팬오션 등 물류 관련 기업은 물류 지원 정책 수혜 예상'] negative_stocks=[] negative_keywords=[] negative_reasons=[]"
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있다. 시푸드 보일 이 대표 메뉴로 준비되고 라이브 스테이션에서 셰프가 직접 원하는 메뉴를 먹기 좋게 잘라 제공한다. 시푸드 보일은 문어와 랍스터 대게 갑오징어 새우 소라 관자 낙지 등 해산물을 쪄낸 뒤 셰프의 비법 시즈닝으로 이국적인 감칠맛을 더한 메뉴다. 프로모션 기간에는 해물전 가리비 불도장 장어 데마끼 로제 해물 뇨끼 등 한식 중식 일식 양식 등 세계 각국의 해산물 메뉴도 즐길 수 있다. 소믈리에 추천 와인 5종과 생맥주를 무제한으로 제공하는 옵션도 선택할 수 있다. 제공되는 와인은 레드와 화이트 와인 각 2종 스파클링 와인 1종으로 취향에 따라 다양하게 즐길 수 있다. 해당 기간 동안 입구 와인셀렉션 코너에서 10만원 이상 와인 구매 시 호텔에서 제작한 주트백도 선물로 증정한다. 이용 가격은 이전과 동일하며 네이버 예약 시 10% 할인 혜택도 제공한다. 주류 무제한 혜택은 2만5000원 추가 시 이용할 수 있다.,https://n.news.naver.com/mnews/article/277/0005112302?sid=101,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행하는데 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있으며 프로모션 기간에는 해물전 가리비 불도장 장어 데마끼 로제 해물 뇨끼 등 한식 중식 일식 양식 등 세계 각국의 해산물 메뉴도 즐길 수 있다.,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있다. 시푸드 보일 이 대표 메뉴로 준비되고 라이브 스테이션에서 셰프가 직접 원하는

In [14]:
# json 변환
# - Pydantic.BaseModel.model_dump() -> dict
# - Pydantic.BaseModel.model_dump_json() -> json_str

# Pydantic 객체를 JSON 문자열로 변환하는 함수
def parse_to_json(obj):
    return obj.model_dump_json()    # StockAnalysis 객체 -> JSON 문자열로 파싱

df['result_json'] = df['result'].apply(parse_to_json)   # result_json 컬럼은 JSON 문자열로 파싱한 결과
df.head()

,date,category,press,title,document,link,summary,content,result,result_json
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,https://n.news.naver.com/mnews/article/052/0001759333?sid=101,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했다.",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,"stock_related=True summary='정부가 수출 확대를 위해 무역금융 규모를 40조 원 이상 확대하고, 물류비 지원과 임시선박 투입 등 수출 중소기업 지원 대책을 추진키로 했습니다. 또한 반도체 등 첨단 산업 육성 전략과 에너지 효율화 방안도 마련해 무역수지 개선에 나설 계획입니다.' positive_stocks=['삼성전자', 'SK하이닉스', '현대글로비스', '팬오션'] positive_keywords=['무역금융 확대', '물류비 지원', '임시선박 투입', '첨단산업 육성', '수출 확대'] positive_reasons=['무역금융 확대와 물류비 절감 대책은 수출 중소기업의 경쟁력 강화에 긍정적 영향', '임시선박 투입으로 물류난 해소 기대', '반도체 등 첨단 산업 육성 전략은 수출 주도 산업 강화와 관련주에 긍정적', '현대글로비스, 팬오션 등 물류 관련 기업은 물류 지원 정책 수혜 예상'] negative_stocks=[] negative_keywords=[] negative_reasons=[]","{""stock_related"":true,""summary"":""정부가 수출 확대를 위해 무역금융 규모를 40조 원 이상 확대하고, 물류비 지원과 임시선박 투입 등 수출 중소기업 지원 대책을 추진키로 했습니다. 또한 반도체 등 첨단 산업 육성 전략과 에너지 효율화 방안도 마련해 무역수지 개선에 나설 계획입니다."",""positive_stocks"":[""삼성전자"",""SK하이닉스"",""현대글로비스"",""팬오션""],""positive_keywords"":[""무역금융 확대"",""물류비 지원"",""임시선박 투입"",""첨단산업 육성"",""수출 확대""],""positive_reasons"":[""무역금융 확대와 물류비 절감 대책은 수출 중소기업의 경쟁력 강화에 긍정적 영향"",""임시선박 투입으로 물류난 해소 기대"",""반도체 등 첨단 산업 육성 전략은 수출 주도 산업 강화와 관련주에 긍정적"",""현대글로비스, 팬오션 등 물류 관련 기업은 물류 지원 정책 수혜 예상""],""negative_stocks"":[],""negative_keywords"":[],""negative_reasons"":[]}"
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있다. 시푸드 보일 이 대표 메뉴로 준비되고 라이브 스테이션에서 셰프가 직접 원하는 메뉴를 먹기 좋게 잘라 제공한다. 시푸드 보일은 문어와 랍스터 대게 갑오징어 새우 소라 관자 낙지 등 해산물을 쪄낸 뒤 셰프의 비법 시즈닝으로 이국적인 감칠맛을 더한 메뉴다. 프로모션 기간에는 해물전 가리비 불도장 장어 데마끼 로제 해물 뇨끼 등 한식 중식 일식 양식 등 세계 각국의 해산물 메뉴도 즐길 수 있다. 소믈리에 추천 와인 5종과 생맥주를 무제한으로 제공하는 옵션도 선택할 수 있다. 제공되는 와인은 레드와 화이트 와인 각 2종 스파클링 와인 1종으로 취향에 따라 다양하게 즐길 수 있다. 해당 기간 동안 입구 와인셀렉션 코너에서 10만원 이상 와인 구매 시 호텔에서 제작한 주트백도 선물로 증정한다. 이용 가격은 이전과 동일하며 네이버 예약 시 10% 할인 혜택도 제공한다. 주류 무제한 혜택은 2만5000원 추가 시 이용할 수 있다.,https://n.news.naver.com/mnews

In [15]:
df['system'] = system_prompt

df = df.rename(columns={
    'content' : 'user',    
    'result_json' : 'assistant'
})

df[['system', 'user', 'assistant']]

,system,user,assistant
0,"# 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n - stock_related를 False로 작성하세요.\n - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n - stock_related를 True로 작성하세요.\n - summary에 뉴스의 요약을 작성하세요.\n - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,"{""stock_related"":true,""summary"":""정부가 수출 확대를 위해 무역금융 규모를 40조 원 이상 확대하고, 물류비 지원과 임시선박 투입 등 수출 중소기업 지원 대책을 추진키로 했습니다. 또한 반도체 등 첨단 산업 육성 전략과 에너지 효율화 방안도 마련해 무역수지 개선에 나설 계획입니다."",""positive_stocks"":[""삼성전자"",""SK하이닉스"",""현대글로비스"",""팬오션""],""positive_keywords"":[""무역금융 확대"",""물류비 지원"",""임시선박 투입"",""첨단산업 육성"",""수출 확대""],""positive_reasons"":[""무역금융 확대와 물류비 절감 대책은 수출 중소기업의 경쟁력 강화에 긍정적 영향"",""임시선박 투입으로 물류난 해소 기대"",""반도체 등 첨단 산업 육성 전략은 수출 주도 산업 강화와 관련주에 긍정적"",""현대글로비스, 팬오션 등 물류 관련 기업은 물류 지원 정책 수혜 예상""],""negative_stocks"":[],""negative_keywords"":[],""negative_reasons"":[]}"
1,"# 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n - stock_related를 False로 작성하세요.\n - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n - stock_related를 True로 작성하세요.\n - summary에 뉴스의 요약을 작성하세요.\n - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있다. 시푸드 보일 이 대표 메뉴로 준비되고 라이브 스테이션에서 셰프가 직접 원하는 메뉴를 먹기 좋게 잘라 제공한다. 시푸드 보일은 문어와 랍스터 대게 갑오징어 새우 소라 관자 낙지 등 해산물을 쪄낸 뒤 셰프의 비법 시즈닝으로 이국적인 감칠맛을 더한 메뉴다. 프로모션 기간에는 해물전 가리비 불도장 장어 데마끼 로제 해물 뇨끼 등 한식 중식 일식 양식 등 세계 각국의 해산물 메뉴도 즐길 수 있다. 소믈리에 추천 와인 5종과 생맥주를 무제한으로 제공하는 옵션도 선택할 수 있다. 제공되는 와인은 레드와 화이트 와인 각 2종 스파클링 와인 1종으로 취향에 따라 다양하게 즐길 수 있다. 해당 기간 동안 입구 와인셀렉션 코너에서 10만원 이상 와인 구매 시 호텔에서 제작한 주트백도 선물로 증정한다. 이용 가격은 이전과 동일하며 네이버 예약 시 10% 할인 혜택도 제공한다. 주류 무제한 혜택은 2만5000원 추가 시 이용할 수 있다.,"{""stock_related"":false,""summary"":""인터컨티넨탈 서울 코엑스 뷔페 레스토랑이 여름 한정으로 '쿨 섬머 페스타'를 진행한다. 주요 메뉴로 시푸드 보일(미국식 해산물 찜)을 선보이고, 문어, 랍스터, 대게 등 다양한 해산물을 활용한다. 7~8월 동안 2만5000원 추가 시 와인 5종과 생맥주를 무제한으로 제공하는 주류 프로모션을 선택할 수 있다. 다양한 해산물 요리와 세계 각국의 메뉴도 즐길 수 있으며, 네이버 예약 할인 혜택과 와인 구매 시 주트백 증정 이벤트도 있다."",""positive_stocks"":[],""positive_keywords"":[],""positive_reasons"":[],""negative_stocks"":[],""negative_keywords"":[],""negative_reasons"":[]}"
2,"# 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n - stock_related를 False로 작성하세요.\n - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n - stock_related를 True

In [18]:
# DataFrame을 JSON 파일로 저장
df[['system', 'user', 'assistant']].to_json(
    'train_json',           #  저장 파일명(경로)
    orient = 'records',     #  리스트 형태로 저장
    force_ascii=False,      #  한글 깨짐 방지
    indent=4                #  들여쓰기 4칸
)

import os 
from datasets import Dataset

dataset = Dataset.from_pandas(df[['system', 'user', 'assistant']])
dataset.push_to_hub(
    'clachic/naver-economy-news2stock'    
)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 118.96ba/s]
Processing Files (1 / 1): 100%|██████████| 93.7kB / 93.7kB, 46.9kB/s  
New Data Upload: 100%|██████████| 93.7kB / 93.7kB, 46.9kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.09s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/clachic/naver-economy-news2stock/commit/4cf134e3ef3d647c6a942163b69135adc91a911b', commit_message='Upload dataset', commit_description='', oid='4cf134e3ef3d647c6a942163b69135adc91a911b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/clachic/naver-economy-news2stock', endpoint='https://huggingface.co', repo_type='dataset', repo_id='clachic/naver-economy-news2stock'), pr_revision=None, pr_num=None)

In [19]:
from datasets import load_dataset

ds = load_dataset("capybaraOh/naver-economy-news2stock")

c:\Users\Playdata\llm\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--capybaraOh--naver-economy-news2stock. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 14907.89 examples/s]


In [21]:
ds.push_to_hub(
    'clachic/naver-economy-news2stock'
)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 43.32ba/s]
Processing Files (1 / 1): 100%|██████████| 1.91MB / 1.91MB, 1.06MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.62s/ shards]
c:\Users\Playdata\llm\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--clachic--naver-economy-news2stock. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as

CommitInfo(commit_url='https://huggingface.co/datasets/clachic/naver-economy-news2stock/commit/2e44a6023124cdfe45b4ab12bef795287c0d0ba9', commit_message='Upload dataset', commit_description='', oid='2e44a6023124cdfe45b4ab12bef795287c0d0ba9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/clachic/naver-economy-news2stock', endpoint='https://huggingface.co', repo_type='dataset', repo_id='clachic/naver-economy-news2stock'), pr_revision=None, pr_num=None)